# E-commerce Journey Analytics

This notebook analyzes 2.76 million real, anonymized RetailRocket events to
support one product decision: which next experiment is most defensible for
increasing purchase conversion?

The SQL model is built from the files under `sql/`; this notebook is the
reader-facing analytical narrative.

## tl;dr

- A 30-minute rule produces **1,761,675 sessions**; **0.81%** contain a
  transaction.
- Returning sessions transact at **1.91%**, compared with **0.53%** for first
  sessions. Excluding the most active 0.1% of visitors reduces the returning
  rate to **1.56%**, so concentration is relevant but does not explain the
  whole difference.
- **31,992 cart sessions** have no same-session transaction. Among first
  observed cart abandoners with complete follow-up, **4.93%** purchase within
  seven days.
- The recommended next step is a **cart-recovery A/B test** for recognized,
  consented visitors. At a 25% relative MDE, it needs **5,416 visitors per arm**
  and roughly **nine calendar weeks**.

These are behavioral associations, not causal effects. The data cannot reveal
why visitors abandon or predict the lift from a reminder.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ecommerce_journey.config import DATABASE_PATH, OUTPUT_DIR, TABLE_DIR
from ecommerce_journey.database import connect
from ecommerce_journey.plots import (
    configure_style,
    plot_cart_recovery,
    plot_funnel,
    plot_retention_heatmap,
    plot_visitor_status,
    plot_weekly_metrics,
)

configure_style()
connection = connect(DATABASE_PATH, read_only=True)
summary = json.loads((OUTPUT_DIR / "summary_metrics.json").read_text(encoding="utf-8"))

summary

{'data': {'raw_event_rows': 2756101,
  'clean_event_rows': 2755641,
  'duplicate_rows_removed': 460,
  'visitors': 1407580,
  'items': 235061,
  'transactions': 17672,
  'min_event_time_utc': '2015-05-03T03:00:04.384000',
  'max_event_time_utc': '2015-09-18T02:59:47.788000'},
 'sessions': {'total_sessions': 1761675,
  'one_session_visitor_share_pct': 87.09607979653022,
  'transaction_session_rate_pct': 0.8115571827947834},
 'funnel': {'view_sessions': 1755781,
  'ordered_view_to_cart_sessions': 35832,
  'ordered_full_funnel_sessions': 10411,
  'ordered_view_to_cart_rate_pct': 2.040801216096996,
  'ordered_cart_to_transaction_rate_pct': 29.055034605938825,
  'ordered_view_to_transaction_rate_pct': 0.5929554995754026,
  'cart_without_transaction_sessions': 31992,
  'same_session_cart_to_transaction_rate_pct': 27.165103360349693},
 'visitor_status': {'new_transaction_session_rate_pct': 0.5345344491964933,
  'returning_transaction_session_rate_pct': 1.9127635239130742,
  'returning_to_new_

## Context & Methods

### Decision

Prioritize one next experiment for an e-commerce product team using the
behavior currently observable in the event log.

### Metric logic

- **Outcome:** share of sessions with at least one transaction.
- **Drivers:** ordered view-to-cart and cart-to-transaction conversion.
- **Longer-term behavior:** return activity by visitor ID in weeks W1-W8.
- **Experiment outcome:** seven-day purchase conversion among assigned,
  eligible cart abandoners.

### Key assumptions

- A session closes after 30 minutes of inactivity.
- Unix timestamps are interpreted in UTC because the source timezone is not
  documented.
- `visitor_id` is an anonymous identifier, not a verified person or account.
- A transaction row is item-level. Purchase conversion uses the presence of a
  transaction, not the number of transaction rows.
- Partial weeks and incomplete future windows are excluded where they would
  bias a comparison.

## Data

The source is version 4 of the
[Retailrocket recommender system dataset](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset),
licensed CC BY-NC-SA 4.0. It records `view`, `addtocart`, and `transaction`
events between May and September 2015.

The pipeline pins the archive and `events.csv` SHA-256 hashes. Raw data are not
committed to Git.

In [2]:
quality = pd.read_csv(TABLE_DIR / "data_quality_summary.csv")
event_types = pd.read_csv(TABLE_DIR / "event_type_summary.csv")

display(quality.T.rename(columns={0: "value"}))
display(event_types)

,value
raw_event_rows,2756101
clean_event_rows,2755641
duplicate_rows_removed,460
duplicate_group_count,458
rows_with_missing_required_fields,0
invalid_event_type_rows,0
transactions_without_id,0
non_transactions_with_id,0
visitors,1407580
items,235061


,event_type,event_rows,visitors,items,transactions
0,view,2664218,1404179,234838,0
1,addtocart,68966,37722,23903,0
2,transaction,22457,11719,12025,17672


There are 460 exact duplicate rows (0.017% of the source), which
are removed before sessionization. Required event fields are complete, all
event types belong to the documented domain, and transaction IDs are populated
only for transaction events.

In [3]:
gap_sensitivity = pd.read_csv(TABLE_DIR / "session_gap_sensitivity.csv")
gap_sensitivity.style.format(
    {
        "cart_session_rate_pct": "{:.3f}%",
        "transaction_session_rate_pct": "{:.3f}%",
    }
)

,gap_minutes,sessions,cart_sessions,transaction_sessions,cart_session_rate_pct,transaction_session_rate_pct
0,15,1809161,46006.000000,15449.000000,2.543%,0.854%
1,30,1761675,43924.000000,14297.000000,2.493%,0.812%
2,60,1726714,42539.000000,13446.000000,2.464%,0.779%


The session count changes with the inactivity threshold, as
expected, but the transaction-session rate ranges only from 0.779% to 0.854%.
The 30-minute definition is therefore consequential for exact counts without
changing the decision-level pattern.

## Results

### 1. Observed funnel

The funnel requires non-decreasing stage timestamps within the same session.
It does not assume that every transaction must have an observed cart event:
persistent carts and missing upstream events can produce shorter paths.

In [4]:
funnel = pd.read_csv(TABLE_DIR / "funnel_summary.csv")
display(
    funnel[
        [
            "view_sessions",
            "ordered_view_to_cart_sessions",
            "ordered_view_to_cart_to_transaction_sessions",
            "cart_without_transaction_sessions",
        ]
    ].T.rename(columns={0: "sessions"})
)
plot_funnel(funnel)

,sessions
view_sessions,1755781
ordered_view_to_cart_sessions,35832
ordered_view_to_cart_to_transaction_sessions,10411
cart_without_transaction_sessions,31992


<Figure size 820x460 with 1 Axes>

Only 2.04% of view sessions contain a later cart event. Once an
ordered cart is observed, 29.06% continue to a transaction in the same
session. The pre-cart loss is much larger, but the log has no search,
recommendation, price, stock, or acquisition context that would make a broad
pre-cart redesign specific enough to test.

### 2. Dynamics and visitor status

In [5]:
weekly = pd.read_csv(TABLE_DIR / "weekly_metrics.csv")
plot_weekly_metrics(weekly)

<Figure size 940x490 with 1 Axes>

The rates fluctuate but show no single structural break across
the 19 complete weeks. This makes a one-date root-cause story unjustified.

In [6]:
visitor_status = pd.read_csv(TABLE_DIR / "visitor_status_sensitivity.csv")
plot_visitor_status(visitor_status)

<Figure size 780x480 with 1 Axes>

In [7]:
concentration = pd.read_csv(TABLE_DIR / "visitor_concentration.csv")
concentration.style.format(
    {
        "session_share_pct": "{:.2f}%",
        "transaction_session_share_pct": "{:.2f}%",
    }
)

,visitor_activity_segment,p999_session_count,visitors,sessions,transaction_sessions,session_share_pct,transaction_session_share_pct
0,remaining_visitors,16,1406313,1713813,12271,97.28%,85.83%
1,top_0_1_percent,16,1267,47862,2026,2.72%,14.17%


Returning sessions are associated with higher purchase intent.
However, this is not a causal benefit of “returning”: visitors self-select into
returning, and the top 0.1% by session count contribute 14.17% of transaction
sessions. Removing them still leaves a 1.56% versus 0.53% difference.

### 3. Retention

In [8]:
retention = pd.read_csv(TABLE_DIR / "cohort_retention.csv")
plot_retention_heatmap(retention)

<Figure size 920x670 with 2 Axes>

In [9]:
activation_retention = pd.read_csv(TABLE_DIR / "activation_retention_w1.csv")
activation_retention.style.format({"w1_retention_rate_pct": "{:.2f}%"})

,first_session_stage,cohort_visitors,retained_w1,w1_retention_rate_pct
0,view_only,1267276,40139,3.17%
1,cart_no_purchase,21347,1514,7.09%
2,purchased,6946,620,8.93%


Weighted W1 retention is 3.26%. First-session cart abandoners
return in W1 at 7.09%, and first-session purchasers at 8.93%, compared with
3.17% for view-only visitors. These groups are defined by behavior, so the
difference is a prioritization signal rather than an estimated effect of
adding to cart or purchasing.

### 4. Cart-recovery opportunity

In [10]:
recovery = pd.read_csv(TABLE_DIR / "cart_recovery_curve.csv")
plot_cart_recovery(recovery)

<Figure size 820x480 with 1 Axes>

Among 27,784 first observed cart abandoners with a complete
seven-day window, 1,369 purchase later. The 4.93% rate is the natural baseline
for sample-size planning. It is not the effect of a reminder.

In [11]:
power = pd.read_csv(TABLE_DIR / "experiment_power.csv")
power_display = power.assign(
    relative_mde_pct=100 * power["relative_mde"],
    absolute_mde_pp=100 * power["absolute_mde"],
    treatment_rate_pct=100 * power["treatment_rate"],
)[
    [
        "relative_mde_pct",
        "absolute_mde_pp",
        "treatment_rate_pct",
        "sample_size_per_arm",
        "estimated_calendar_weeks",
    ]
]
power_display.style.format(
    {
        "relative_mde_pct": "{:.0f}%",
        "absolute_mde_pp": "{:.2f}",
        "treatment_rate_pct": "{:.2f}%",
        "sample_size_per_arm": "{:,.0f}",
        "estimated_calendar_weeks": "{:.0f}",
    }
)

,relative_mde_pct,absolute_mde_pp,treatment_rate_pct,sample_size_per_arm,estimated_calendar_weeks
0,15%,0.74,5.67%,"14,414",21
1,20%,0.99,5.91%,"8,286",13
2,25%,1.23,6.16%,"5,416",9
3,30%,1.48,6.41%,"3,839",7


A 25% relative MDE balances sensitivity and feasibility:
5,416 visitors per arm, about 7.3 weeks of enrollment at the observed inflow,
plus the seven-day outcome window. The planning duration is nine weeks.

The proposed experiment randomizes each recognized, consented visitor once
after the first qualifying cart-abandonment session. The primary metric is
seven-day purchase conversion by intention to treat. Delivery, opt-out,
complaint, cancellation, refund, and margin metrics are required guardrails.
See `docs/experiment_design.md` for the complete protocol.

## Takeaways

1. **Run a cart-recovery experiment only after confirming contactability and
   adding assignment/delivery logging.** The audience is specific, measurable,
   and high-intent relative to view-only visitors.
2. **Instrument checkout before diagnosing checkout friction.** Add
   `checkout_started`, `payment_failed`, and `order_completed` rather than
   treating every missing transaction as the same cause.
3. **Do not claim causal lift from returning status or cart behavior.** The
   segments are selected by observed behavior and differ in latent intent.
4. **Keep the current recommendation provisional.** Identity loss,
   cross-device behavior, channel, price, stock, promotion, revenue, and order
   quality are unavailable and could change the decision.

The analysis supports what to test next; only randomized evidence can support
whether the intervention should ship.